In [ ]:
import requests
import os
from dotenv import load_dotenv    
load_dotenv()

api_key = os.getenv("SARVAM_API_KEY")
MODEL = os.getenv("SARVAM_MODEL")
BASE_URL = os.getenv("SARVAM_API_URL")

In [ ]:
class SarvamSTT:
    def __init__(self, api_key):
        self.api_key = api_key
        self.base_url = BASE_URL

    def transcribe_audio(self, audio_file_path, model=MODEL, language_code="unknown", with_timestamps=False):
        if not os.path.exists(audio_file_path):
            raise FileNotFoundError(f"Audio file not found: {audio_file_path}")
        if os.path.getsize(audio_file_path) == 0:
            raise ValueError("Audio file is empty")
        
        # Prepare headers and form data
        headers = {"api-subscription-key": self.api_key}
        files = {"file": (os.path.basename(audio_file_path), open(audio_file_path, "rb"), "audio/wav")}
        data = {
            "model": model,
            "language_code": language_code,
            "with_timestamps": str(with_timestamps).lower()
        }

        try:
            response = requests.post(
                self.base_url,
                headers=headers,
                files=files,
                data=data
            )
            response.raise_for_status() 
            return response.json()
        except requests.exceptions.RequestException as e:
            raise requests.exceptions.RequestException(f"Sarvam API error: {str(e)}")
        finally:
            files["file"][1].close()

In [ ]:
def main():
    api_key = os.getenv("SARVAM_API_KEY") 
    if not api_key:
        raise ValueError("SARVAM_API_KEY environment variable not set")
    
    stt = SarvamSTT(api_key)
    
    audio_file = "/Users/mukeshyadav/Desktop/Projects/deeplearning-genai/data/test_audio.mp3" 
    try:
        result = stt.transcribe_audio(audio_file, language_code="en-IN")
        print("Transcription:", result.get("transcript", ""))
        
        result_with_timestamps = stt.transcribe_audio(audio_file, language_code="en-IN", with_timestamps=True)
        print("Transcription with timestamps:", result_with_timestamps)
    except Exception as e:
        print(f"Error: {str(e)}")

if __name__ == "__main__":
    main()